# SmokeWatch Replay: Full-Period Data Pipeline

This notebook expands the validated five-day feasibility workflow to the complete Dixie Fire pilot period from July 13 through August 31, 2021.

The first step creates a complete UTC county-hour backbone. This structural grid ensures that zero fire detections remain distinct from missing PM2.5 or weather measurements.

In [8]:
from pathlib import Path

import pandas as pd
import numpy as np

# Locate the repository by searching the current directory
# and its parent directories for the Git repository.
current_directory = Path.cwd().resolve()
project_root = None

for candidate_directory in [
    current_directory,
    *current_directory.parents,
]:
    if (
        (candidate_directory / ".git").exists()
        and (candidate_directory / "notebooks").is_dir()
        and (candidate_directory / "data").is_dir()
    ):
        project_root = candidate_directory
        break

if project_root is None:
    raise RuntimeError(
        "Could not locate the smokewatch-replay project root."
    )

pilot_counties = [
    "Butte",
    "Lassen",
    "Plumas",
    "Shasta",
    "Tehama",
]

pilot_start_utc = pd.Timestamp(
    "2021-07-13 00:00:00",
    tz="UTC",
)

pilot_end_utc = pd.Timestamp(
    "2021-08-31 23:00:00",
    tz="UTC",
)

expected_hour_count = 1200
expected_county_count = 5
expected_county_hour_rows = 6000

full_period_interim_directory = (
    project_root / "data" / "interim" / "full_period"
)
full_period_interim_directory.mkdir(
    parents=True,
    exist_ok=True,
)

county_hour_grid_path = (
    full_period_interim_directory
    / "dixie_county_hour_grid_2021-07-13_2021-08-31.csv"
)

print("Notebook working directory:", current_directory)
print("Project root:", project_root)
print("Pilot start:", pilot_start_utc)
print("Pilot end:", pilot_end_utc)
print("Pilot counties:", pilot_counties)
print("Expected hours:", expected_hour_count)
print("Expected county-hour rows:", expected_county_hour_rows)

Notebook working directory: /Users/VV/Documents/GitHub/smokewatch-replay/notebooks/notebooks
Project root: /Users/VV/Documents/GitHub/smokewatch-replay
Pilot start: 2021-07-13 00:00:00+00:00
Pilot end: 2021-08-31 23:00:00+00:00
Pilot counties: ['Butte', 'Lassen', 'Plumas', 'Shasta', 'Tehama']
Expected hours: 1200
Expected county-hour rows: 6000


In [9]:
full_period_hours = pd.date_range(
    start=pilot_start_utc,
    end=pilot_end_utc,
    freq="h",
)

full_county_hour_grid = (
    pd.MultiIndex.from_product(
        [full_period_hours, pilot_counties],
        names=["event_hour_utc", "county"],
    )
    .to_frame(index=False)
    .sort_values(["event_hour_utc", "county"])
    .reset_index(drop=True)
)

print("UTC hours generated:", len(full_period_hours))
print("Counties:", len(pilot_counties))
print("County-hour rows:", len(full_county_hour_grid))

display(full_county_hour_grid.head(10))
display(full_county_hour_grid.tail(10))

UTC hours generated: 1200
Counties: 5
County-hour rows: 6000


,event_hour_utc,county
0,2021-07-13 00:00:00+00:00,Butte
1,2021-07-13 00:00:00+00:00,Lassen
2,2021-07-13 00:00:00+00:00,Plumas
3,2021-07-13 00:00:00+00:00,Shasta
4,2021-07-13 00:00:00+00:00,Tehama
5,2021-07-13 01:00:00+00:00,Butte
6,2021-07-13 01:00:00+00:00,Lassen
7,2021-07-13 01:00:00+00:00,Plumas
8,2021-07-13 01:00:00+00:00,Shasta
9,2021-07-13 01:00:00+00:00,Tehama


,event_hour_utc,county
5990,2021-08-31 22:00:00+00:00,Butte
5991,2021-08-31 22:00:00+00:00,Lassen
5992,2021-08-31 22:00:00+00:00,Plumas
5993,2021-08-31 22:00:00+00:00,Shasta
5994,2021-08-31 22:00:00+00:00,Tehama
5995,2021-08-31 23:00:00+00:00,Butte
5996,2021-08-31 23:00:00+00:00,Lassen
5997,2021-08-31 23:00:00+00:00,Plumas
5998,2021-08-31 23:00:00+00:00,Shasta
5999,2021-08-31 23:00:00+00:00,Tehama


In [10]:
rows_per_county = (
    full_county_hour_grid
    .groupby("county", as_index=False)
    .agg(
        county_hour_rows=("event_hour_utc", "size"),
        unique_hours=("event_hour_utc", "nunique"),
        earliest_hour=("event_hour_utc", "min"),
        latest_hour=("event_hour_utc", "max"),
    )
)

rows_per_hour = (
    full_county_hour_grid
    .groupby("event_hour_utc")
    .size()
)

continuous_hourly_sequence = (
    full_period_hours
    .to_series()
    .diff()
    .dropna()
    .eq(pd.Timedelta(hours=1))
    .all()
)

grid_validation = pd.DataFrame(
    [
        (
            "Total county-hour rows",
            len(full_county_hour_grid),
            expected_county_hour_rows,
        ),
        (
            "Unique UTC hours",
            full_county_hour_grid["event_hour_utc"].nunique(),
            expected_hour_count,
        ),
        (
            "Unique counties",
            full_county_hour_grid["county"].nunique(),
            expected_county_count,
        ),
        (
            "Duplicate county-hour keys",
            full_county_hour_grid.duplicated(
                subset=["event_hour_utc", "county"]
            ).sum(),
            0,
        ),
        (
            "Invalid timestamps",
            full_county_hour_grid["event_hour_utc"].isna().sum(),
            0,
        ),
        (
            "Minimum county rows per hour",
            rows_per_hour.min(),
            expected_county_count,
        ),
        (
            "Maximum county rows per hour",
            rows_per_hour.max(),
            expected_county_count,
        ),
        (
            "Minimum hours per county",
            rows_per_county["county_hour_rows"].min(),
            expected_hour_count,
        ),
        (
            "Maximum hours per county",
            rows_per_county["county_hour_rows"].max(),
            expected_hour_count,
        ),
        (
            "Continuous hourly sequence",
            continuous_hourly_sequence,
            True,
        ),
        (
            "Timestamp timezone is UTC",
            str(
                full_county_hour_grid[
                    "event_hour_utc"
                ].dt.tz
            ),
            "UTC",
        ),
    ],
    columns=["check", "observed", "expected"],
)

grid_validation["status"] = grid_validation.apply(
    lambda row: (
        "PASS"
        if row["observed"] == row["expected"]
        else "FAIL"
    ),
    axis=1,
)

display(grid_validation)
display(rows_per_county)

failed_grid_checks = grid_validation[
    grid_validation["status"] != "PASS"
]

assert failed_grid_checks.empty, (
    "The full-period grid failed validation:\n"
    + failed_grid_checks.to_string(index=False)
)

assert (
    full_county_hour_grid["event_hour_utc"].min()
    == pilot_start_utc
)

assert (
    full_county_hour_grid["event_hour_utc"].max()
    == pilot_end_utc
)

print("Full-period county-hour grid validation passed.")

,check,observed,expected,status
0,Total county-hour rows,6000,6000,PASS
1,Unique UTC hours,1200,1200,PASS
2,Unique counties,5,5,PASS
3,Duplicate county-hour keys,0,0,PASS
4,Invalid timestamps,0,0,PASS
5,Minimum county rows per hour,5,5,PASS
6,Maximum county rows per hour,5,5,PASS
7,Minimum hours per county,1200,1200,PASS
8,Maximum hours per county,1200,1200,PASS
9,Continuous hourly sequence,True,True,PASS


,county,county_hour_rows,unique_hours,earliest_hour,latest_hour
0,Butte,1200,1200,2021-07-13 00:00:00+00:00,2021-08-31 23:00:00+00:00
1,Lassen,1200,1200,2021-07-13 00:00:00+00:00,2021-08-31 23:00:00+00:00
2,Plumas,1200,1200,2021-07-13 00:00:00+00:00,2021-08-31 23:00:00+00:00
3,Shasta,1200,1200,2021-07-13 00:00:00+00:00,2021-08-31 23:00:00+00:00
4,Tehama,1200,1200,2021-07-13 00:00:00+00:00,2021-08-31 23:00:00+00:00


Full-period county-hour grid validation passed.


In [11]:
full_county_hour_grid.to_csv(
    county_hour_grid_path,
    index=False,
)

reloaded_county_hour_grid = pd.read_csv(
    county_hour_grid_path
)

reloaded_county_hour_grid["event_hour_utc"] = (
    pd.to_datetime(
        reloaded_county_hour_grid["event_hour_utc"],
        utc=True,
        errors="coerce",
    )
)

assert len(reloaded_county_hour_grid) == 6000

assert (
    reloaded_county_hour_grid["event_hour_utc"]
    .isna()
    .sum()
    == 0
)

assert (
    reloaded_county_hour_grid.duplicated(
        subset=["event_hour_utc", "county"]
    ).sum()
    == 0
)

assert (
    reloaded_county_hour_grid["county"].nunique()
    == 5
)

assert (
    reloaded_county_hour_grid["event_hour_utc"].nunique()
    == 1200
)

pd.testing.assert_frame_equal(
    full_county_hour_grid,
    reloaded_county_hour_grid,
)

print("Full-period grid saved and reloaded successfully.")
print("Output path:", county_hour_grid_path)
print("Reloaded rows:", len(reloaded_county_hour_grid))
print("Reloaded columns:", len(reloaded_county_hour_grid.columns))

Full-period grid saved and reloaded successfully.
Output path: /Users/VV/Documents/GitHub/smokewatch-replay/data/interim/full_period/dixie_county_hour_grid_2021-07-13_2021-08-31.csv
Reloaded rows: 6000
Reloaded columns: 2


## Full-Period Structural Backbone Validation

The complete pilot period contains 1,200 UTC hours across five counties, producing 6,000 unique county-hour rows.

This grid will serve as the structural backbone for the full-period NASA FIRMS, EPA AirData, and NOAA GHCNh datasets. Each source will be validated independently before it is joined to this backbone. Missing measurements will remain explicitly missing, while valid zero-event records will remain zero.